# Lección 9.2 — Capas y profundidad: por qué apilar funciona

> **Módulo 9 · Redes neuronales desde la intuición** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 9.1 (la neurona artificial), Lección 4.4 (sobreajuste vs. subajuste)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía por qué **varias capas** de neuronas resuelven lo que una sola no puede (el XOR).
- Entender una capa oculta como un **cambio de representación**: la red inventa sus propias características.
- Explicar por qué apilar capas **sin activación** no sirve de nada.
- Contar los **parámetros** de una red y predecir el efecto de cambiar el número de **capas** y de **neuronas**.
- Construir una red multicapa (MLP) en PyTorch con `nn.Sequential`.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import make_circles, make_moons, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

np.random.seed(0)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo · dispositivo:", device)

---
## 1. 📖 La intuición: un equipo de especialistas

En la Lección 9.1 vimos que una neurona sola es como **una persona que suma evidencias y decide**. Y vimos su límite: solo traza **una recta**, así que no puede con el XOR.

Piensa ahora en cómo decide un **banco** si un movimiento con tarjeta es fraude:

```
 Capa 1 (especialistas)                 Capa 2 (gerente)
 ┌──────────────────────────────┐
 │ 🕵️ "¿Monto raro para ESTE     │──┐
 │     cliente?"                 │  │
 │ 🌍 "¿País raro para él?"      │──┼──►  Combina los informes  ──►  ¿Fraude?
 │ 🕒 "¿Hora rara para él?"      │──┘
 └──────────────────────────────┘
```

Cada especialista mira los datos crudos y responde **una pregunta sencilla**. El gerente no mira los datos crudos: decide a partir de **los informes** de los especialistas. Ninguno es muy listo por separado, pero el equipo resuelve un problema que nadie resolvería solo.

**Una red neuronal es exactamente eso:** capas de neuronas donde cada capa trabaja con lo que le entrega la anterior.

### ✋ Ejercicio 1 — Calentamiento
En la Lección 9.1, la neurona del parque de diversiones solo funcionó cuando **tú** le diste la característica `edad²` ("¿edad extrema?"). En la analogía del banco, ¿qué papel jugaste tú en ese ejercicio?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Hiciste de **especialista** (de capa 1): inventaste un informe útil ("edad extrema") y la neurona hizo de **gerente**. La promesa de las capas es que la red **contrate a sus propios especialistas**: que aprenda sola qué características intermedias le sirven.
</details>

---
## 2. 📖 El concepto: resolver el XOR con dos capas

Vamos a resolver el XOR (salida 1 cuando las entradas son **distintas**) con un equipo de **2 especialistas + 1 gerente**, eligiendo los pesos a mano. Usamos la neurona escalón de la Lección 9.1.

| Neurona | Pesos | Sesgo | Qué pregunta responde |
|---|---|---|---|
| Especialista `h1` | [1, 1] | −0.5 | "¿Hay **al menos una** encendida?" (OR) |
| Especialista `h2` | [1, 1] | −1.5 | "¿Están **las dos** encendidas?" (AND) |
| Gerente | [1, −1] (sobre `h1`, `h2`) | −0.5 | "¿Al menos una, **pero no** las dos?" |

### ✋ Ejercicio 2 — Haz de red neuronal
Completa la tabla a mano (recuerda: la salida es 1 si `z > 0`).

| x1 | x2 | h1 (OR) | h2 (AND) | salida = escalón(h1 − h2 − 0.5) |
|---|---|---|---|---|
| 0 | 0 | ? | ? | ? |
| 0 | 1 | ? | ? | ? |
| 1 | 0 | ? | ? | ? |
| 1 | 1 | ? | ? | ? |

✍️ *Tu tabla:* luego comprueba con la celda.

In [ ]:
def escalon(z):
    return (z > 0).astype(int)

X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

# Capa 1: dos especialistas (cada columna de W1 es una neurona)
W1 = np.array([[1, 1],
               [1, 1]])
b1 = np.array([-0.5, -1.5])
H = escalon(X_xor @ W1 + b1)             # informes de los especialistas

# Capa 2: el gerente
W2 = np.array([1, -1])
b2 = -0.5
salida = escalon(H @ W2 + b2)

for x, h, s in zip(X_xor, H, salida):
    print(f"x = {x} → h1, h2 = {h} → salida = {s}")

<details><summary><b>👀 Ver respuesta</b></summary>

| x1 | x2 | h1 | h2 | salida |
|---|---|---|---|---|
| 0 | 0 | 0 | 0 | 0 − 0 − 0.5 < 0 → **0** |
| 0 | 1 | 1 | 0 | 1 − 0 − 0.5 > 0 → **1** |
| 1 | 0 | 1 | 0 | **1** |
| 1 | 1 | 1 | 1 | 1 − 1 − 0.5 < 0 → **0** |

¡Exactamente el XOR! Lo que una neurona no podía, **tres neuronas en dos capas** lo hacen sin esfuerzo.
</details>

### 📖 El secreto: la capa oculta **cambia el punto de vista**
¿Por qué funciona? Mira los cuatro puntos **como los ve el gerente**: no en las coordenadas originales `(x1, x2)`, sino en las coordenadas `(h1, h2)` que le entregan los especialistas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
etiquetas = [0, 1, 1, 0]
axes[0].scatter(*X_xor.T, c=etiquetas, cmap="coolwarm", s=300, edgecolor="k")
axes[0].set_title("Lo que ve la entrada (x1, x2)\nninguna recta separa los colores")
axes[1].scatter(*(H + np.random.default_rng(0).normal(0, .03, H.shape)).T,   # pequeño temblor para ver los puntos repetidos
                c=etiquetas, cmap="coolwarm", s=300, edgecolor="k")
axes[1].plot([0.2, 1.2], [-0.2, 0.8], "k--", label="h1 − h2 = 0.5")
axes[1].set_title("Lo que ve el gerente (h1, h2)\n¡ahora una recta basta!"); axes[1].legend()
for ax in axes:
    ax.set_xlim(-0.3, 1.3); ax.set_ylim(-0.3, 1.3); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

En el espacio `(h1, h2)`, los dos puntos rojos (`[0,1]` y `[1,0]`) caen **en el mismo lugar**, `(1, 0)`, y los azules quedan en `(0, 0)` y `(1, 1)`. Ahora una sola recta los separa, y el gerente (que es solo una neurona) puede hacerlo.

> 🔑 **Idea central del deep learning:** cada capa **transforma los datos** en una nueva representación donde el problema es más fácil. A las capas intermedias se les llama **capas ocultas** (*hidden layers*) porque no vemos directamente lo que producen: son "informes internos".

### 🔮 Ejercicio 3
Si cambias el sesgo del gerente de `−0.5` a `+0.5`, ¿qué tabla de verdad calculará la red? ✏️ Pruébalo cambiando `b2` en la celda del Ejercicio 2.

✍️ *Mi predicción:*

<details><summary><b>👀 Ver respuesta</b></summary>

Con `b2 = +0.5`: `[0,0]` → 0 − 0 + 0.5 > 0 → **1**; `[0,1]` y `[1,0]` → 1 + 0.5 → **1**; `[1,1]` → 1 − 1 + 0.5 → **1**. ¡Siempre dice 1! El gerente se volvió "fácil de convencer". Con los **mismos especialistas**, cambiar solo la última neurona cambia por completo la función: las capas son **piezas reutilizables**.
</details>

---
## 3. 📖 Cómo funciona: una red multicapa (MLP)

Una red con una o más capas ocultas donde **cada neurona se conecta con todas** las de la capa anterior se llama **perceptrón multicapa** (**MLP**, *multilayer perceptron*) o red **densa** / *fully connected*:

```
 entrada (2)     oculta (4)      salida (1)
    x1 ─────┬───►  ○  ───┐
            ├───►  ○  ───┤
    x2 ─────┼───►  ○  ───┼───►  ○  ──► predicción
            └───►  ○  ───┘
```

En PyTorch se arma con piezas apiladas en `nn.Sequential`:
- `nn.Linear(entradas, salidas)` → una **capa** completa de neuronas (todas las sumas ponderadas + sesgos a la vez).
- `nn.ReLU()` → la **activación** de esas neuronas. ReLU es la más usada hoy: deja pasar los positivos y convierte los negativos en 0 (la veremos a fondo en la Lección 9.3).

In [ ]:
red = nn.Sequential(
    nn.Linear(2, 4),   # capa oculta: 2 entradas → 4 neuronas
    nn.ReLU(),         # activación de la capa oculta
    nn.Linear(4, 1),   # capa de salida: 4 informes → 1 neurona
)
print(red)

### 📖 Contar parámetros
Cada neurona tiene **un peso por entrada + un sesgo**. Una capa `nn.Linear(entradas, salidas)` tiene, por tanto:

`parámetros = entradas × salidas + salidas`

### ✋ Ejercicio 4 — Cuenta a mano
¿Cuántos parámetros tiene la red de arriba (`2 → 4 → 1`)? ¿Y una red `2 → 8 → 8 → 1`?

✍️ *Mis cuentas:*

In [ ]:
def contar_parametros(modelo):
    return sum(p.numel() for p in modelo.parameters())

red_grande = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 8), nn.ReLU(), nn.Linear(8, 1))
print("2 → 4 → 1     :", contar_parametros(red), "parámetros")
print("2 → 8 → 8 → 1 :", contar_parametros(red_grande), "parámetros")

<details><summary><b>👀 Ver respuesta</b></summary>

- `2 → 4 → 1`: (2·4 + 4) + (4·1 + 1) = 12 + 5 = **17**.
- `2 → 8 → 8 → 1`: (2·8 + 8) + (8·8 + 8) + (8·1 + 1) = 24 + 72 + 9 = **105**.

Las ReLU no tienen parámetros: no se "aprenden". Fíjate en que la capa del medio (8 → 8) es la más cara: los parámetros crecen con el **producto** de los tamaños de capas vecinas.
</details>

### 📖 Ahora que la red aprenda sola
En el XOR elegimos los pesos a mano. Con datos reales la red debe **descubrirlos**. Usemos un conjunto con forma de **dos círculos concéntricos**: imposible de separar con una recta.

La función `entrenar_red` arma una red con las capas ocultas que le pidas y la entrena (el bucle lo explicaremos en la Lección 9.4; el optimizador `Adam`, en la 9.5).

In [ ]:
X_c, y_c = make_circles(n_samples=400, noise=0.1, factor=0.4, random_state=0)
Xc_t = torch.tensor(X_c, dtype=torch.float32)
yc_t = torch.tensor(y_c, dtype=torch.float32).unsqueeze(1)

def crear_red(entradas, capas_ocultas, salidas=1, activacion=nn.ReLU):
    capas, n_ant = [], entradas
    for n in capas_ocultas:                      # p. ej. [8, 8] = dos capas ocultas de 8
        capas += [nn.Linear(n_ant, n), activacion()]
        n_ant = n
    capas.append(nn.Linear(n_ant, salidas))      # capa de salida
    return nn.Sequential(*capas)

def entrenar_red(X, y, capas_ocultas, pasos=1000, lr=0.01, semilla=0, activacion=nn.ReLU):
    torch.manual_seed(semilla)
    modelo = crear_red(X.shape[1], capas_ocultas, activacion=activacion)
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)
    perdida_fn = nn.BCEWithLogitsLoss()          # entropía cruzada binaria; incluye la sigmoide (Lección 9.3)
    for _ in range(pasos):
        perdida = perdida_fn(modelo(X), y)
        opt.zero_grad(); perdida.backward(); opt.step()
    return modelo

def exactitud(modelo, X, y):
    with torch.no_grad():
        return ((modelo(X) > 0).float() == y).float().mean().item()

def dibujar_frontera(modelo, X, y, ax, titulo):
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - .5, X[:, 0].max() + .5, 200),
                         np.linspace(X[:, 1].min() - .5, X[:, 1].max() + .5, 200))
    with torch.no_grad():
        zz = modelo(torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)).numpy()
    ax.contourf(xx, yy, (zz > 0).reshape(xx.shape), alpha=.3, cmap="coolwarm")
    ax.scatter(*X.T, c=y, cmap="coolwarm", edgecolor="k", s=12)
    ax.set_title(titulo, fontsize=10); ax.set_xticks([]); ax.set_yticks([])

plt.scatter(*X_c.T, c=y_c, cmap="coolwarm", edgecolor="k", s=15); plt.title("Dos círculos"); plt.show()

### 🔮 Ejercicio 5 — Predice antes de ejecutar
Vamos a entrenar cuatro redes: **sin capa oculta** (una sola neurona = logística), y con una capa oculta de **2**, **8** y **64** neuronas.
1. ¿Qué exactitud esperas sin capa oculta?
2. ¿Cuántas neuronas crees que hacen falta para "encerrar" el círculo interior?

✍️ *Mis predicciones:*

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, ocultas in zip(axes, [[], [2], [8], [64]]):
    modelo = entrenar_red(Xc_t, yc_t, ocultas)
    nombre = "sin capa oculta" if not ocultas else f"1 capa oculta de {ocultas[0]}"
    dibujar_frontera(modelo, X_c, y_c, ax,
                     f"{nombre}\nexactitud {exactitud(modelo, Xc_t, yc_t):.0%} · {contar_parametros(modelo)} parámetros")
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Sin capa oculta ≈ 50 %**: una recta parte el plano en dos y deja la mitad de cada color de cada lado. Es como tirar una moneda.
2. Con **2 neuronas** la red mejora (≈ 78 %) pero solo puede formar una "cuña" con dos rectas. Con **8** ya encierra el círculo (≈ 100 %), y con **64** también, con un borde más redondeado.

Cada neurona oculta aporta **una recta**; el gerente las combina. Necesitas al menos 3 rectas para encerrar una zona (un triángulo), y con más rectas el borde se parece más a un círculo.
</details>

### ✏️ Ejercicio 6 — Mira dentro de la capa oculta
¿Qué aprendieron los especialistas? Entrenamos una red con **3** neuronas ocultas y dibujamos, para cada una, **dónde se enciende** (su salida ReLU > 0). ✏️ Cambia `N_OCULTAS` a 2 o 5 y observa.

In [ ]:
N_OCULTAS = 3    # ✏️ prueba 2 o 5

modelo = entrenar_red(Xc_t, yc_t, [N_OCULTAS])
print(f"Exactitud con {N_OCULTAS} neuronas ocultas: {exactitud(modelo, Xc_t, yc_t):.0%}")

xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 200), np.linspace(-1.5, 1.5, 200))
malla = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
with torch.no_grad():
    ocultas = modelo[1](modelo[0](malla)).numpy()       # salida de la capa oculta (tras ReLU)
fig, axes = plt.subplots(1, N_OCULTAS, figsize=(3.5 * N_OCULTAS, 3.3))
for i, ax in enumerate(np.atleast_1d(axes)):
    ax.contourf(xx, yy, (ocultas[:, i] > 0).reshape(xx.shape), alpha=.4, cmap="Greens")
    ax.scatter(*X_c.T, c=y_c, cmap="coolwarm", s=6)
    ax.set_title(f"neurona oculta {i + 1}: zona verde = encendida", fontsize=9); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Cada neurona oculta divide el plano con **una recta** (verde de un lado, apagada del otro). Con 3 neuronas, las tres rectas forman un **triángulo** alrededor del círculo interior y la exactitud llega a ≈ 100 %. El gerente aprende algo como "si ninguna de las tres está muy encendida, estás dentro del triángulo → círculo interior". Con 2 neuronas no alcanza para cerrar la zona.

Nadie le dijo a la red que usara triángulos: **lo inventó** para resolver el problema.
</details>

---
## 4. 📖 ¿Qué pasa si… apilo capas pero **olvido la activación**?

Parece que más capas = más poder. Pero hay una trampa: si entre las capas **no hay activación**, apilar no sirve de nada. Una capa lineal hace "multiplicar y sumar"; aplicar otra encima es **otra vez multiplicar y sumar**, y el resultado de las dos juntas es… una sola capa lineal.

> Analogía: si conviertes pesos a dólares y luego dólares a euros, es lo mismo que convertir pesos a euros directamente con **un único** tipo de cambio. Encadenar conversiones lineales no crea nada nuevo.

### 🔮 Ejercicio 7 — Predice antes de ejecutar
Entrenamos con los círculos una red **profunda** de 3 capas de 64 neuronas, pero **sin activaciones** (solo `nn.Linear` apiladas). ¿Qué exactitud esperas: ≈ 50 %, ≈ 75 % o ≈ 100 %?

✍️ *Mi predicción:*

In [ ]:
torch.manual_seed(0)
red_sin_activacion = nn.Sequential(nn.Linear(2, 64), nn.Linear(64, 64), nn.Linear(64, 64), nn.Linear(64, 1))
opt = torch.optim.Adam(red_sin_activacion.parameters(), lr=0.01)
for _ in range(1000):
    perdida = nn.BCEWithLogitsLoss()(red_sin_activacion(Xc_t), yc_t)
    opt.zero_grad(); perdida.backward(); opt.step()

print(f"Parámetros: {contar_parametros(red_sin_activacion):,}")
print(f"Exactitud : {exactitud(red_sin_activacion, Xc_t, yc_t):.0%}")

# Comprobación: las 4 capas equivalen a UNA sola capa lineal 2 → 1
with torch.no_grad():
    W, b = torch.eye(2), torch.zeros(2)
    for capa in red_sin_activacion:
        W, b = capa.weight @ W, capa.weight @ b + capa.bias      # componemos las capas
    una_capa = Xc_t @ W.T + b
    print("¿Misma salida que una sola capa lineal?", torch.allclose(una_capa, red_sin_activacion(Xc_t), atol=1e-4))

<details><summary><b>👀 Ver respuesta</b></summary>

**≈ 50 %**, como una sola neurona, a pesar de sus más de 8,000 parámetros. La comprobación lo confirma: la red entera equivale **exactamente** a una capa lineal `2 → 1` (con sus 3 parámetros). Toda la "profundidad" era una ilusión.

👉 **Las activaciones no lineales son las que dan poder a las capas.** Sin ellas, cualquier red, por profunda que sea, solo traza rectas. Por eso la Lección 9.3 está dedicada a ellas.
</details>

---
## 5. 📖 Hiperparámetros: ¿cuántas capas y cuántas neuronas?

La **arquitectura** de un MLP se decide con dos perillas:

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| Neuronas por capa (**ancho**) | Cuántos "especialistas" hay en cada capa | Fronteras más detalladas; más parámetros → riesgo de **sobreajuste** y más lento | Red más simple → riesgo de **subajuste** | 32–256 (potencias de 2 por costumbre) |
| Número de capas ocultas (**profundidad**) | Cuántos niveles de "informes sobre informes" | Puede combinar ideas en niveles; más difícil de entrenar (9.7) | Más simple y estable | 1–3 para datos tabulares |
| Activación | La forma de la "decisión" de cada neurona | — | — | ReLU (Lección 9.3) |

### 📖 ¿Cuántas neuronas bastan? El teorema de aproximación universal
Un resultado famoso dice que **una sola capa oculta con suficientes neuronas puede aproximar casi cualquier función** tan bien como quieras. La intuición es sencilla con ReLU: cada neurona aporta un "quiebre" en la curva, y la suma de muchos tramos rectos puede imitar cualquier curva (como dibujar un círculo con muchos segmentos cortos).

### 🔮 Ejercicio 8 — Dibuja una ola con tramos rectos
Le pedimos a una red con **una** capa oculta que imite la curva `sin(2x)`. Con **2**, **6** y **50** neuronas ReLU: ¿cuántos "quiebres" esperas ver en cada caso?

✍️ *Mi predicción:*

In [ ]:
x = torch.linspace(-3, 3, 200).unsqueeze(1)
y_ola = torch.sin(2 * x)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, n in zip(axes, [2, 6, 50]):           # ✏️ prueba otros tamaños
    torch.manual_seed(0)
    modelo = nn.Sequential(nn.Linear(1, n), nn.ReLU(), nn.Linear(n, 1))
    opt = torch.optim.Adam(modelo.parameters(), lr=0.01)
    for _ in range(3000):
        perdida = nn.MSELoss()(modelo(x), y_ola)
        opt.zero_grad(); perdida.backward(); opt.step()
    ax.plot(x, y_ola, "k:", label="objetivo")
    ax.plot(x, modelo(x).detach(), lw=2, label="red")
    ax.set_title(f"{n} neuronas ocultas · error {perdida.item():.3f}"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **2 neuronas:** como mucho 2 quiebres: una línea quebrada que apenas sigue la tendencia (error ≈ 0.5).
- **6 neuronas:** ya sigue varias subidas y bajadas con tramos rectos visibles, pero aún se le escapa parte de la ola (error ≈ 0.1).
- **50 neuronas:** casi idéntica a la ola (error ≈ 0.000).

Es la aproximación universal en acción. Pero ojo: el teorema dice que **existe** una red así, no que sea fácil de entrenar ni que **generalice** bien con pocos datos.
</details>

### 📖 Ancho vs. profundidad
Si una capa basta en teoría, ¿para qué apilar varias? Porque en la práctica **la profundidad es más eficiente**: cada capa reutiliza lo que construyó la anterior. En imágenes, por ejemplo, la primera capa detecta bordes, la siguiente combina bordes en formas, la siguiente formas en partes (ojo, rueda) y la última partes en objetos (lo verás en el Módulo 10). Para hacer lo mismo con una sola capa harían falta **muchísimas** más neuronas.

| | Red **ancha** y poco profunda | Red **estrecha** y profunda |
|---|---|---|
| Idea | Muchos especialistas, un solo nivel | Pocos por nivel, muchos niveles de "informes sobre informes" |
| Ventaja | Fácil de entrenar | Construye ideas complejas a partir de simples con menos parámetros |
| Riesgo | Muchos parámetros para patrones complejos | Gradientes que se desvanecen (Lección 9.7) |

### ✋ Ejercicio 9
Una red `10 → 100 → 1` y otra `10 → 20 → 20 → 20 → 1`. ¿Cuál tiene **más parámetros**? Haz la cuenta.

✍️ *Mi cuenta:*

<details><summary><b>👀 Ver respuesta</b></summary>

- `10 → 100 → 1`: (10·100 + 100) + (100 + 1) = 1,100 + 101 = **1,201**.
- `10 → 20 → 20 → 20 → 1`: 220 + 420 + 420 + 21 = **1,081**.

La red profunda tiene **menos** parámetros y, sin embargo, tiene tres niveles para combinar ideas. Puedes verificarlo con `contar_parametros(crear_red(10, [100]))` y `contar_parametros(crear_red(10, [20, 20, 20]))`.
</details>

---
## 6. 📖 Errores comunes y señales de alerta

Más neuronas no es gratis. Una red con miles de parámetros y pocos datos puede **memorizar** el ruido, igual que el árbol sin límites de la Lección 5.4.

### 🔮 Ejercicio 10 — La red gigante con pocos datos
Tenemos solo **80 ejemplos** de entrenamiento (dos lunas con bastante ruido) y 1,000 de prueba. Comparamos una red pequeña (`[8]`) con una gigante (`[256, 256]`). ¿Cuál gana en **entrenamiento**? ¿Y en **prueba**?

✍️ *Mi predicción:*

In [ ]:
X_l, y_l = make_moons(n_samples=1080, noise=0.4, random_state=1)
X_l_tr, y_l_tr = torch.tensor(X_l[:80], dtype=torch.float32), torch.tensor(y_l[:80], dtype=torch.float32).unsqueeze(1)
X_l_te, y_l_te = torch.tensor(X_l[80:], dtype=torch.float32), torch.tensor(y_l[80:], dtype=torch.float32).unsqueeze(1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, ocultas in zip(axes, [[8], [256, 256]]):
    m = entrenar_red(X_l_tr, y_l_tr, ocultas, pasos=2000)
    dibujar_frontera(m, X_l[:80], y_l[:80], ax,
                     f"capas ocultas {ocultas} · {contar_parametros(m):,} parámetros\n"
                     f"train {exactitud(m, X_l_tr, y_l_tr):.0%} · test {exactitud(m, X_l_te, y_l_te):.0%}")
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- La red **gigante** (≈ 67,000 parámetros para 80 ejemplos) llega al **100 %** en entrenamiento con una frontera retorcida que rodea puntos sueltos, pero en prueba baja a ≈ 77 %: **sobreajuste**.
- La red **pequeña** no es perfecta en entrenamiento (≈ 85 %), pero en prueba se mantiene en ≈ 83 %: le gana a la gigante con 2,000 veces menos parámetros.

Las redes tienen herramientas propias contra el sobreajuste (dropout, weight decay…) que verás en la **Lección 9.8**.
</details>

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Red profunda que rinde igual que una logística | Faltan activaciones entre capas (todo es lineal) | Poner `nn.ReLU()` después de cada `nn.Linear` oculta |
| Exactitud baja en entrenamiento y prueba | Red demasiado pequeña (**subajuste**) | Más neuronas o una capa más |
| Entrenamiento ≈ 100 % y prueba mucho menor | Demasiados parámetros para tan pocos datos (**sobreajuste**) | Red más pequeña, más datos, regularización (9.8) |
| Error de forma: `mat1 and mat2 shapes cannot be multiplied` | Las salidas de una capa no coinciden con las entradas de la siguiente | Revisar que `nn.Linear(a, b)` vaya seguida de `nn.Linear(b, c)` |
| Poner una ReLU **después de la última capa** | La salida no puede ser negativa (mal para regresión y para logits) | La capa de salida va **sin** ReLU (Lección 9.3) |

---
## 7. 📋 Ficha del modelo

| | Perceptrón multicapa (MLP / red densa) |
|---|---|
| **Tipo** | Supervisado: clasificación y regresión |
| **Intuición** | Capas de "especialistas": cada capa crea características nuevas a partir de las de la anterior |
| **Cómo aprende** | Descenso de gradiente sobre la pérdida; los gradientes se calculan con retropropagación (9.4) |
| **Hiperparámetros clave** | Número de capas, neuronas por capa, activación, tasa de aprendizaje (9.6) |
| **✅ Fortalezas** | Aproxima casi cualquier función · Aprende sus propias características · Base de todo el deep learning |
| **❌ Debilidades** | Necesita datos y escalar entradas · Muchos hiperparámetros · Poco interpretable · Sobreajusta con pocos datos |
| **Úsalo cuando…** | Hay relaciones no lineales complejas y bastantes datos; datos como imágenes, audio o texto (con arquitecturas especializadas, M10–M12) |
| **Evítalo cuando…** | Datos tabulares pequeños o medianos: los ensambles de árboles suelen ganar con menos esfuerzo (Lección 6.6) |

## 8. 🧠 Resumen en 5 puntos
1. Una red neuronal es un **equipo de neuronas en capas**: cada capa trabaja con los "informes" de la anterior.
2. Las **capas ocultas cambian la representación** de los datos hasta que el problema se vuelve fácil (el XOR se vuelve separable).
3. **Sin activaciones no lineales**, apilar capas no sirve: todo colapsa en una sola capa lineal.
4. Una capa `nn.Linear(a, b)` tiene `a·b + b` parámetros; **más ancho y más profundidad** = más capacidad y más riesgo de sobreajuste.
5. Una capa oculta basta **en teoría** (aproximación universal), pero la **profundidad** es más eficiente en la práctica.

---
## 9. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 9.1 Repaso rápido (opción múltiple)

**1.** Una red tiene la arquitectura `4 → 5 → 3`. ¿Cuántos parámetros tiene?
- a) 12 · b) 35 · c) 43 · d) 60

**2.** Apilas 10 capas `nn.Linear` **sin** activaciones entre ellas. La red resultante es equivalente a…
- a) Una red de 10 capas muy potente
- b) Una sola capa lineal
- c) Un árbol de decisión
- d) Depende de los datos

**3.** ¿Por qué se llaman "ocultas" las capas ocultas?
- a) Porque están desactivadas al predecir
- b) Porque sus salidas son representaciones internas que no vemos en los datos ni en la predicción
- c) Porque no tienen parámetros
- d) Porque solo se usan en entrenamiento

**4.** Tu MLP tiene 99 % en entrenamiento y 70 % en prueba con 200 ejemplos. ¿Qué cambio de arquitectura tiene **más** sentido?
- a) Pasar de `[128, 128]` a `[512, 512, 512]`
- b) Pasar de `[128, 128]` a `[16]`
- c) Quitar las activaciones
- d) Añadir una ReLU a la salida

**5.** El teorema de aproximación universal dice que…
- a) Cualquier red entrenada generaliza bien
- b) Una capa oculta con suficientes neuronas puede aproximar casi cualquier función
- c) Las redes profundas siempre ganan a los árboles
- d) Con ReLU no hay sobreajuste

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** (4·5 + 5) + (5·3 + 3) = 25 + 18 = **43**.
2. **b)** Encadenar transformaciones lineales da otra transformación lineal.
3. **b)** Son "informes internos" de la red.
4. **b)** Es sobreajuste: hay que **reducir** la capacidad. a) lo empeora, c) la vuelve lineal (subajuste), d) es un error.
5. **b)** Habla de lo que una red **puede representar**, no de lo fácil que sea entrenarla ni de cómo generaliza.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** En el XOR elegimos los pesos a mano y la red tenía 9 parámetros. Si la entrenaras desde cero muchas veces con semillas distintas, ¿crees que encontraría **siempre los mismos** pesos que pusimos nosotros? ¿Importa?

**B. Caso de análisis 💼** Una aseguradora tiene una tabla con 5,000 pólizas y 20 columnas (edad, tipo de auto, siniestros previos…). Un consultor propone "una red neuronal profunda de 10 capas de 512 neuronas porque el deep learning es lo más moderno". ¿Qué le responderías? Usa lo que sabes de parámetros, sobreajuste y de la Lección 6.6.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Casi seguro que **no**: hay muchísimas combinaciones de pesos que resuelven el XOR (por ejemplo, intercambiar los dos especialistas, o escalar todos los pesos). Y **no importa**: nos interesa la función que calcula la red, no los valores concretos. Por eso los pesos de una red rara vez se interpretan uno a uno (a diferencia de una logística).

**B.** Esa red tendría unos **2.4 millones** de parámetros (≈ 9 capas de 512·512) para 5,000 ejemplos: receta para el **sobreajuste**, además de lenta y difícil de entrenar. En datos tabulares de ese tamaño, los ensambles de árboles (Random Forest, Gradient Boosting) suelen ganar con menos esfuerzo (Lección 6.6). Si se quiere probar una red, empezar con algo pequeño (1–2 capas de 32–128 neuronas), compararla con un buen ensamble usando validación cruzada y quedarse con lo que funcione, no con lo más moderno.
</details>

### 9.3 🧩 Práctica integradora: "Lector de dígitos escritos a mano"

Una oficina de correos quiere leer automáticamente los dígitos de los códigos postales. Tienes **1,797 imágenes** de 8×8 píxeles de dígitos (0–9) escritos a mano. Construirás tu primer MLP de varias clases en 6 pasos cortos. **No avances sin revisar cada respuesta.**

**Paso 1 ▶️ — Mira los datos.**

In [ ]:
digitos = load_digits()
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, etiqueta in zip(axes, digitos.images, digitos.target):
    ax.imshow(img, cmap="gray_r"); ax.set_title(etiqueta); ax.axis("off")
plt.show()
print("Forma de los datos:", digitos.data.shape)

✋ ¿Cuántas **entradas** tendrá la red? ¿Y cuántas **salidas**, si queremos una puntuación por cada dígito posible?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Cada imagen tiene 8×8 = **64 píxeles** → 64 entradas (la imagen se "aplana" en una fila de 64 números). Y **10 salidas**, una por dígito (0–9): la red dará 10 puntuaciones y elegiremos la más alta. La forma de los datos, `(1797, 64)`, lo confirma.
</details>

**Paso 2 ▶️ — Prepara los datos.** Separamos entrenamiento y prueba, y **escalamos** (Lección 2.3) usando solo el entrenamiento.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(digitos.data, digitos.target, test_size=0.3,
                                          stratify=digitos.target, random_state=0)
escalador = StandardScaler().fit(X_tr)
Xd_tr = torch.tensor(escalador.transform(X_tr), dtype=torch.float32)
Xd_te = torch.tensor(escalador.transform(X_te), dtype=torch.float32)
yd_tr, yd_te = torch.tensor(y_tr), torch.tensor(y_te)       # para varias clases, etiquetas enteras 0–9
print(f"Entrenamiento: {len(Xd_tr)} · prueba: {len(Xd_te)}")

**Paso 3 🔮 — Línea base.** Una regresión logística multiclase equivale a una red **sin capas ocultas**. ¿Qué exactitud esperas en prueba: ≈ 70 %, ≈ 85 % o ≈ 95 %?

✍️ *Mi apuesta:*

In [ ]:
base = LogisticRegression(max_iter=2000).fit(Xd_tr.numpy(), y_tr)
print(f"Regresión logística en prueba: {base.score(Xd_te.numpy(), y_te):.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

≈ **97 %**. Sorpresa: los dígitos de 8×8 son bastante "lineales" y una logística ya es muy buena. ¡Una red tendrá que esforzarse para ganarle! Siempre conviene medir la línea base antes de presumir de deep learning (Lección 4.7).
</details>

**Paso 4 ✏️ — Tu primer MLP de 10 clases.** Usamos `crear_red` con 64 entradas y 10 salidas. Para varias clases, la pérdida es `nn.CrossEntropyLoss` (la verás en la Lección 9.3). Primero cuenta los parámetros de `64 → 32 → 10`, luego ejecuta.

✍️ *Mi cuenta:*

In [ ]:
CAPAS_OCULTAS = [32]          # ✏️ prueba [], [8], [128], [128, 64]

def entrenar_digitos(capas_ocultas, pasos=300, semilla=0):
    torch.manual_seed(semilla)
    modelo = crear_red(64, capas_ocultas, salidas=10)
    opt = torch.optim.Adam(modelo.parameters(), lr=0.01)
    for _ in range(pasos):
        perdida = nn.CrossEntropyLoss()(modelo(Xd_tr), yd_tr)
        opt.zero_grad(); perdida.backward(); opt.step()
    with torch.no_grad():
        acc_tr = (modelo(Xd_tr).argmax(1) == yd_tr).float().mean().item()
        acc_te = (modelo(Xd_te).argmax(1) == yd_te).float().mean().item()
    return modelo, acc_tr, acc_te

mlp, acc_tr, acc_te = entrenar_digitos(CAPAS_OCULTAS)
print(f"Capas ocultas {CAPAS_OCULTAS}: {contar_parametros(mlp):,} parámetros · "
      f"train {acc_tr:.1%} · test {acc_te:.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

`64 → 32 → 10`: (64·32 + 32) + (32·10 + 10) = 2,080 + 330 = **2,410** parámetros. Entrenamiento **100 %** y prueba ≈ **97 %**: empata con la logística. En un problema casi lineal, la capa oculta no aporta mucho.
</details>

**Paso 5 ✋ — Compara arquitecturas.** Ejecuta la tabla y responde: ¿la red más grande es la mejor en prueba? ¿Qué pasa con `[2]`?

In [ ]:
for ocultas in [[], [2], [8], [32], [128], [128, 64]]:
    m, a_tr, a_te = entrenar_digitos(ocultas)
    print(f"{str(ocultas):<10} {contar_parametros(m):>6,} parámetros · train {a_tr:6.1%} · test {a_te:6.1%}")

✍️ *Mis observaciones:*

<details><summary><b>👀 Ver respuesta</b></summary>

- `[2]` es un **cuello de botella**: obliga a resumir toda la imagen en solo 2 números, y la exactitud cae mucho (≈ 55 % en prueba, ¡peor que la logística!). Es **subajuste** causado por la arquitectura, no por los datos.
- A partir de `[8]` todas rondan el **94–98 %** en prueba, y las grandes no ganan claramente a las medianas aunque tienen 5–6 veces más parámetros. Pasado cierto tamaño, **más neuronas no ayudan**: el límite está en los datos.
- `[]` (sin capa oculta, como la logística) ya da ≈ 96 %.

Las diferencias de 1 punto entre las mejores pueden ser ruido (con 540 imágenes de prueba, cada imagen vale ≈ 0.2 puntos). Para decidir en serio usarías validación (Lección 4.6), no el conjunto de prueba.
</details>

**Paso 6 ✋ — Para el gerente.** Explica en **dos frases**, sin jerga, qué hace la capa oculta de tu red de dígitos.

✍️ *Mi explicación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo: *"La primera capa son 32 detectores que aprenden solos a reconocer trazos útiles (una curva arriba, una línea vertical a la izquierda…). La última capa combina qué detectores se encendieron para decidir qué dígito es."* No le dijimos qué trazos buscar: los descubrió a partir de los ejemplos. En el **Módulo 10** verás redes (CNN) diseñadas a propósito para detectar trazos en imágenes.
</details>

---
## 📚 Para profundizar
- 🎮 **Interactivo:** *TensorFlow Playground* (playground.tensorflow.org): resuelve "círculo" y "XOR" añadiendo neuronas y capas, y mira qué dibuja cada neurona oculta.
- 📺 **Video:** 3Blue1Brown, "¿Qué es una red neuronal?" (capítulo 1 de *Deep Learning*, con subtítulos en español).
- 📘 **Lectura:** Michael Nielsen, *Neural Networks and Deep Learning*, capítulo 4: "Una prueba visual de que las redes pueden calcular cualquier función".

---
⬅️ **Anterior:** [9.1 La neurona artificial: suma ponderada + decisión](9.1-la-neurona-artificial.ipynb) · ➡️ **Siguiente:** [9.3 Funciones de activación: sigmoide, tanh, ReLU y familia](9.3-funciones-de-activacion.ipynb)